# 1.1 — Autoregressive Sequence Model from Scratch

**Mechanism of the day:** the chain rule — turn one impossible distribution into
`L` easy ones, and learn each with a tiny causal transformer.

Notebook 0.2 steered a *dumb* generator: every residue drawn independently from
background frequencies. It knew nothing about what makes a sequence a sequence,
so "control" had nothing good to work with. Time to build a real `p(x)`.

Every autoregressive model rests on one exact identity:

```
p(x_1 ... x_L) = p(x_1) * p(x_2 | x_1) * p(x_3 | x_1 x_2) * ... = prod_t p(x_t | x_<t)
```

This is **not** an approximation — it is the product rule, always true. You cannot
write down a table over `20^28` sequences, but you *can* learn one function that
maps a prefix to a distribution over the next 20 letters, and apply it `L` times.
That is the whole trick. It is what GPT does to text and what ProGen-style
protein language models do to amino acids.

What the factorization buys you:
- **exact likelihood** — so *perplexity* is a real, comparable number (unlike in
  diffusion, where you only get a bound),
- **trivial sampling** — one token at a time, left to right,
- **a temperature knob** — the same reward-vs-diversity dial you met in 0.2.

What it costs you:
- generation is **sequential** — `L` forward passes, no parallel shortcut,
- every token sees only its **left** context (rungs 1.2 and 1.5 attack exactly this).

**How to use this notebook:** read the concept, implement the reps (they
`raise NotImplementedError`), make the checkpoints (`assert`s) pass. Solutions are
at the very bottom — don't peek until you've tried. Everything here trains on a
laptop CPU in well under a minute.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (5.6, 3.4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

AA = 'ACDEFGHIKLMNPQRSTVWY'
N_AA = len(AA)
BOS = N_AA                 # 'begin sequence' token -> index 20
VOCAB = N_AA + 1           # 21 tokens the model can emit
STOI = {a: i for i, a in enumerate(AA)}
HYDRO = 'AILMFV'           # our hydrophobic set

# Natural amino-acid frequencies (UniProt, %) — the same table as notebook 0.2.
FREQ = {'A':8.25,'R':5.53,'N':4.06,'D':5.46,'C':1.38,'Q':3.93,'E':6.72,'G':7.07,
        'H':2.27,'I':5.91,'L':9.66,'K':5.80,'M':2.41,'F':3.86,'P':4.74,'S':6.65,
        'T':5.36,'W':1.10,'Y':2.92,'V':6.87}
f = np.array([FREQ[a] for a in AA])
is_h = np.array([a in HYDRO for a in AA])

L = 28            # 4 heptads
EPS = 0.05        # 5% of slots break the rule (nothing in biology is clean)
CORE = (0, 3)     # heptad positions 'a' and 'd' — the hydrophobic face
print('vocab', VOCAB, '| length', L, '| hydrophobic set', HYDRO)

## Part 1 — a dataset with a secret

We need data with structure worth learning. Real proteins have plenty, but we want
something whose answer key we *keep*, so we can prove what the model did and did
not learn. So we synthesize it.

Our corpus: cartoon **amphipathic helices**. An alpha-helix (the one you built in
0.1) turns ~100 degrees per residue, so it comes back around about every **7**
residues — a *heptad*. Helices that pack against a hydrophobic core or lie on a
membrane put their greasy residues on **one face**: heptad positions **a** and
**d**. Everything else faces solvent and is polar.

So each sequence is generated as: pick a phase, make positions where
`(i - phase) % 7` is `0` or `3` hydrophobic (95% of the time), everything else
polar (95% of the time). Within a group, letters follow the natural frequencies
above — `L` is a common hydrophobic, `W` is rare — rather than being uniform.

The crucial detail: **the phase is drawn at random per sequence and never
recorded**. It is a hidden variable. That single choice is what makes this dataset
worth a transformer.

In [ ]:
# Emission distributions. Core slots: 95% hydrophobic. Other slots: 95% polar.
# Within a group, letters follow natural frequencies (NOT uniform) — this is what
# will give temperature something to sharpen, later on.
p_core = np.where(is_h, f * (1 - EPS) / f[is_h].sum(), f * EPS / f[~is_h].sum())
p_non  = np.where(is_h, f * EPS / f[is_h].sum(),  f * (1 - EPS) / f[~is_h].sum())

def make_corpus(n, L=L):
    '''n cartoon amphipathic helices, each with its own RANDOM hidden phase.'''
    phases = rng.integers(0, 7, size=n)
    out = np.zeros((n, L), dtype=np.int64)
    for i in range(L):
        r = (i - phases) % 7
        core = (r == CORE[0]) | (r == CORE[1])
        for mask, p in ((core, p_core), (~core, p_non)):
            k = int(mask.sum())
            if k:
                out[mask, i] = rng.choice(N_AA, size=k, p=p)
    return out, phases

def decode(row):
    '''int array -> amino-acid string.'''
    return ''.join(AA[i] for i in row)

def full_xy(data):
    '''Teacher-forcing pair for a WHOLE dataset (used for evaluation).'''
    x = np.concatenate([np.full((len(data), 1), BOS, dtype=np.int64), data[:, :-1]], 1)
    return torch.from_numpy(x), torch.from_numpy(data.copy())

train, train_phase = make_corpus(8000)
val, val_phase = make_corpus(1000)
print('train', train.shape, ' val', val.shape, '\n')
for r, ph in zip(train[:4], train_phase[:4]):
    s = decode(r)
    print(s, ' ', ''.join('H' if c in HYDRO else '.' for c in s), ' phase=%d' % ph)

In [ ]:
# Sort rows by their hidden phase and the register becomes obvious.
pat = np.array([[c in HYDRO for c in decode(r)] for r in train[:60]])
order = np.argsort(train_phase[:60])

plt.figure(figsize=(7.2, 3.2))
plt.imshow(pat[order], aspect='auto', interpolation='nearest',
           cmap=ListedColormap(['#eaeef4', BLUE]))
plt.xlabel('position'); plt.ylabel('sequence (sorted by hidden phase)')
plt.title('hydrophobic residues sit on a period-7 register')
plt.show()
print('Sorted by phase -> clean diagonal stripes.')
print('The model never sees the phase; it has to infer it from the prefix. ✓')

### Why this dataset defeats the easy models

Because the phase is random per sequence, **position alone tells you nothing**.
Position 5 is hydrophobic in some sequences and polar in others, in exactly the
proportion you'd expect by chance. A model with only a positional embedding is
stuck at the unconditional letter frequencies.

A **bigram** — look back exactly one token — is stuck too. The register repeats
every 7 residues, so the evidence it needs lives 7 steps back, not 1. (It gets a
crumb: if the previous residue was hydrophobic, this one probably isn't, because
`a` and `d` are never adjacent. Watch for that crumb in the numbers.)

To predict well you must **look back across the whole prefix**, infer the hidden
phase from where the hydrophobics landed, and carry it forward. That is precisely
the job self-attention does — and we are about to measure exactly how much it buys.

## Part 2 — teacher forcing: `L` training signals from one sequence

The naive reading of the chain rule is "train `L` separate models." You don't. One
model handles all positions at once, because of a trick called **teacher forcing**:

- feed the model `x = [BOS, s_1, s_2, ..., s_{L-1}]`
- ask it to predict `y = [s_1, s_2, s_3, ..., s_L]`

Position `t` of the input holds token `t-1`, so predicting `y[t]` from `x[:t+1]` is
exactly the conditional `p(s_t | s_<t)`. Every position is a training example, and
a causal mask (Part 3) is what stops position `t` from cheating by looking at the
answer sitting further right in the same row.

`BOS` is a dedicated 21st token that means "nothing yet", giving the model
something to condition on when predicting the very first residue.

### Rep 1 — `make_batch(data, batch_size)`
Sample `batch_size` random sequences and build the teacher-forcing pair.
Return `int64` torch tensors of shape `[batch_size, L]`, where `x[:, 0] == BOS`
and `x[:, t] == y[:, t-1]`.

In [ ]:
def make_batch(data, batch_size):
    '''Sample batch_size sequences -> (x, y) teacher-forcing tensors [B, L] int64.'''
    # YOUR CODE HERE
    # hint: rng.integers picks the rows; np.concatenate prepends the BOS column;
    #       torch.from_numpy converts (use .copy() so you own the memory)
    raise NotImplementedError

# --- checkpoint ---
xb, yb = make_batch(train, 8)
assert xb.shape == (8, L) and yb.shape == (8, L), 'wrong shape'
assert xb.dtype == torch.int64 and yb.dtype == torch.int64, 'need int64 tensors'
assert (xb[:, 0] == BOS).all(), 'every input row must start with BOS'
assert (xb[:, 1:] == yb[:, :-1]).all(), 'x must be y shifted right by one'
print('x[0] =', xb[0].tolist()[:9], '...')
print('y[0] =', yb[0].tolist()[:9], '...')
print('teacher forcing ✓ — one sequence, L supervised predictions, all in parallel')

## Part 3 — a baseline, and the number that judges everything

Before any transformer, build the dumbest honest autoregressive model: a
**bigram**. Estimate `p(next | prev)` by counting pairs in the training set. It is
a legitimate `p(x)` — it factorizes by the chain rule just like a transformer —
it simply has a tiny memory.

We score every model with **perplexity**:

```
perplexity = exp( mean over tokens of -log p(true token | context) )
```

Read it as *"as confused as if it were choosing uniformly among this many
letters."* Lower is better. Our vocab is 21, so 21 means "learned nothing."
Add-alpha smoothing keeps unseen pairs from producing `log 0`.

### Rep 2 — `fit_bigram(data, alpha=1.0)`
Count every `(prev, next)` pair, add `alpha` to every cell, and normalize each row.
Return a `[VOCAB, VOCAB]` array whose rows are probability distributions.
Use `full_xy(data)` to get the aligned pairs.

In [ ]:
def fit_bigram(data, alpha=1.0):
    '''Count-based P(next | prev) with add-alpha smoothing -> [VOCAB, VOCAB].'''
    # YOUR CODE HERE
    # hint: start from np.full((VOCAB, VOCAB), alpha), then np.add.at(counts, (prev, nxt), 1.0)
    raise NotImplementedError

# --- checkpoint ---
P = fit_bigram(train)
assert P.shape == (VOCAB, VOCAB), 'wrong shape'
assert np.allclose(P.sum(axis=1), 1.0), 'every row must be a distribution'
xv, yv = full_xy(val)
bigram_nll = -np.log(P[xv.numpy().ravel(), yv.numpy().ravel()]).mean()
bigram_ppl = math.exp(bigram_nll)
assert bigram_ppl < VOCAB, 'a bigram should at least beat uniform'
print('bigram val perplexity %.2f   (uniform guessing would be %d)' % (bigram_ppl, VOCAB))

### The answer key: what is the *best possible* perplexity?

Here is something almost no tutorial can show you. With real data nobody knows the
optimal loss, so you never learn whether 3.1 nats is brilliant or terrible. But we
*wrote* this generative process, so we can compute the floor exactly.

The best conceivable predictor sees the prefix, forms a **Bayesian posterior over
the 7 possible phases**, and mixes the corresponding emission distributions:

```
p(x_t | x_<t) = sum over phi of  P(phi | x_<t) * P(x_t | phi, t)
```

Only 7 phases, so this is exact, not an estimate. **No model can beat it** — it is
the irreducible noise we baked in with `EPS` and the letter frequencies. Your
transformer's job is to get close to it.

In [ ]:
def bayes_nll(data):
    '''Exact optimal predictor: marginalize the 7 phases with a Bayesian posterior.'''
    n, Ln = data.shape
    tab = np.stack([np.stack([p_core if ((t - phi) % 7) in CORE else p_non
                              for t in range(Ln)]) for phi in range(7)])   # [7, L, 20]
    logpost = np.full((n, 7), -math.log(7))          # uniform prior over phase
    total = 0.0
    for t in range(Ln):
        post = np.exp(logpost - logpost.max(1, keepdims=True))
        post /= post.sum(1, keepdims=True)
        pred = np.einsum('np,pv->nv', post, tab[:, t, :])     # mix over phases
        xt = data[:, t]
        total += -np.log(pred[np.arange(n), xt]).sum()
        logpost += np.log(tab[:, t, :][:, xt].T)              # update the posterior
    return total / (n * Ln)

# Two reference points that need no data at all.
H_marg = -( ((2/7) * p_core + (5/7) * p_non) * np.log((2/7) * p_core + (5/7) * p_non) ).sum()
H_known = (2/7) * -(p_core * np.log(p_core)).sum() + (5/7) * -(p_non * np.log(p_non)).sum()

BAYES_PPL = math.exp(bayes_nll(val))
print('uniform guess          ppl %5.2f   (learned nothing)' % VOCAB)
print('letter frequencies     ppl %5.2f   (unigram: knows L is common, W is rare)' % math.exp(H_marg))
print('your bigram            ppl %5.2f' % bigram_ppl)
print('BAYES-OPTIMAL          ppl %5.2f   <- nothing can beat this' % BAYES_PPL)
print('  (if the phase were handed to you: ppl %.2f)' % math.exp(H_known))

## Part 4 — causal self-attention, the actual mechanism

Now the engine. Self-attention lets position `t` build its prediction by **looking
at every earlier position and deciding how much each one matters**:

```
scores = Q K^T / sqrt(d_head)     # [T, T] — how much does query i want key j?
mask   = j > i -> -inf            # THE autoregressive constraint
weights = softmax(scores)         # each row sums to 1
out    = weights @ V              # a weighted average of earlier value vectors
```

Three things worth internalizing:

- **`sqrt(d_head)`** keeps the dot products from growing with dimension, which
  would saturate the softmax into a one-hot and kill gradients.
- **The mask is the entire autoregressive property.** Set it to `-inf` *before* the
  softmax so those positions get exactly zero weight. Mask after, and the rows no
  longer sum to 1 — a classic, silent bug.
- **Getting this wrong leaks the future**, and the symptom is seductive: training
  loss dives, val perplexity looks impossible, and samples are garbage. Our
  checkpoint tests for the leak directly, by perturbing a future token and checking
  that earlier outputs don't move.

### Rep 3 — `causal_self_attention(q, k, v)`
`q, k, v` are `[B, n_head, T, d_head]`. Return `(out, att)` with `out` the same
shape as `v` and `att` the `[B, n_head, T, T]` weights (rows summing to 1, zero
above the diagonal). Everything is batched — plain `@` broadcasts over the leading
dimensions, so no loops.

In [ ]:
def causal_self_attention(q, k, v):
    '''Scaled dot-product attention with a causal mask. Returns (out, att).'''
    # YOUR CODE HERE
    # hint: k.transpose(-2, -1) for the scores; build the mask with
    #       torch.tril(torch.ones(T, T, dtype=torch.bool)); Tensor.masked_fill(~mask, float('-inf'))
    raise NotImplementedError

# --- checkpoint ---
q, k, v = (torch.randn(2, 4, L, 16) for _ in range(3))
out, att = causal_self_attention(q, k, v)
assert out.shape == (2, 4, L, 16), 'out must match v'
assert att.shape == (2, 4, L, L), 'att must be [B, n_head, T, T]'
assert torch.allclose(att.sum(-1), torch.ones(2, 4, L), atol=1e-5), 'rows must sum to 1'
assert (att.triu(1) == 0).all(), 'no attention weight may sit above the diagonal'

# The real test: move the LAST value vector. Every earlier output must be unchanged.
v2 = v.clone(); v2[:, :, -1, :] += 99.0
out2, _ = causal_self_attention(q, k, v2)
assert torch.allclose(out[:, :, :-1, :], out2[:, :, :-1, :], atol=1e-6), 'the future leaked!'
assert not torch.allclose(out[:, :, -1, :], out2[:, :, -1, :]), 'the last position should react'
print('causal attention ✓ — position t provably cannot see t+1')

### The rest of the transformer (given)

Standard pre-norm decoder blocks, wrapped around **your** attention function. Note
how little there is: multi-head is just "split the channels into `n_head` groups,
attend within each, concatenate"; a block is `x + attn(norm(x))` then
`x + mlp(norm(x))`. Residual stream in, residual stream out.

The model is ~105k parameters — smaller than a single attention head of GPT-2.

In [ ]:
class MultiHeadAttention(nn.Module):
    '''Split channels into heads, run YOUR causal attention, concatenate.'''
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head, self.d_head = n_head, d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)     # all three projections at once
        self.proj = nn.Linear(d_model, d_model)
        self.last_att = None

    def forward(self, x):
        B, T, Cc = x.shape
        q, k, v = self.qkv(x).split(Cc, dim=2)
        split = lambda z: z.view(B, T, self.n_head, self.d_head).transpose(1, 2)
        o, att = causal_self_attention(split(q), split(k), split(v))
        self.last_att = att.detach()                   # kept for the plot later
        return self.proj(o.transpose(1, 2).contiguous().view(B, T, Cc))

class Block(nn.Module):
    '''Pre-norm transformer block: attention then MLP, both residual.'''
    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_head)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                 nn.Linear(4 * d_model, d_model))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class TinyAR(nn.Module):
    def __init__(self, vocab=VOCAB, d_model=64, n_head=4, n_layer=2, block=L):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(block, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head) for _ in range(n_layer)])
        self.lnf = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)          # -> logits over the vocab

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for b in self.blocks:
            x = b(x)
        return self.head(self.lnf(x))                  # [B, T, VOCAB]

model = TinyAR()
print('parameters:', sum(p.numel() for p in model.parameters()))

## Part 5 — training

The loss is cross-entropy between the logits at every position and the true next
token — i.e. exactly the mean negative log-likelihood whose exponential is
perplexity. Flatten `[B, T, VOCAB]` to `[B*T, VOCAB]` and let `F.cross_entropy`
handle all `B*T` predictions at once.

Watch the val curve against the two reference lines. Beating the bigram means the
model found *something* multi-step; approaching the Bayes line means it recovered
essentially all of the recoverable structure.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
xv_t, yv_t = full_xy(val)
hist = []
t0 = time.time()

for step in range(1, 601):
    xb, yb = make_batch(train, 64)
    loss = F.cross_entropy(model(xb).reshape(-1, VOCAB), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0 or step == 1:
        model.eval()
        with torch.no_grad():
            vl = F.cross_entropy(model(xv_t).reshape(-1, VOCAB), yv_t.reshape(-1)).item()
        model.train()
        hist.append((step, loss.item(), vl))
        if step % 200 == 0 or step == 1:
            print('step %3d  train %.4f  val %.4f  (val ppl %.2f)' % (step, loss.item(), vl, math.exp(vl)))
print('trained in %.1fs on CPU' % (time.time() - t0))

h = np.array(hist)
fig, ax = plt.subplots()
ax.plot(h[:, 0], h[:, 1], color=BLUE, lw=2, label='train')
ax.plot(h[:, 0], h[:, 2], color=GREEN, lw=2, label='val')
ax.axhline(bigram_nll, color=INK, ls=':', lw=1.5)
ax.axhline(math.log(BAYES_PPL), color=INK, ls='--', lw=1.5)
ax.set_xlim(-15, 790)                                   # room for the labels
lbl = dict(va='center', fontsize=9, color=INK, bbox=dict(fc='white', ec='none', pad=1.5))
ax.text(625, bigram_nll, 'bigram', **lbl)
ax.text(625, math.log(BAYES_PPL), 'Bayes floor', **lbl)
ax.set_xlabel('step'); ax.set_ylabel('loss (nats / token)')
ax.set_title('the model closes almost all of the gap to optimal')
ax.grid(alpha=.15); ax.legend(frameon=False)
plt.show()

### Rep 4 — `perplexity(model, data)`
Run the model over a whole dataset (no gradients) and return
`exp(mean cross-entropy)`. Remember `model.eval()` and `torch.no_grad()`.

In [ ]:
def perplexity(model, data):
    '''Exponentiated mean NLL per token over an entire dataset.'''
    # YOUR CODE HERE
    # hint: full_xy(data) gives (x, y); F.cross_entropy already returns the MEAN nll
    raise NotImplementedError

# --- checkpoint ---
ppl = perplexity(model, val)
assert ppl < bigram_ppl, 'the transformer must beat the bigram'
assert ppl < 15.5, 'expected ~13; if much higher, training or attention is off'
print('perplexity   transformer %.2f  |  bigram %.2f  |  uniform %d' % (ppl, bigram_ppl, VOCAB))
print('Bayes floor  %.2f    gap remaining: %.2f' % (BAYES_PPL, ppl - BAYES_PPL))
print('\nIt cannot beat the floor — that residual is the noise we baked in with EPS. ✓')

## Part 6 — sampling, and the temperature knob

Sampling an autoregressive model is a loop: feed what you have, take the logits at
the **last** position, turn them into probabilities, draw one token, append, repeat.

**Temperature** divides the logits before the softmax:

- `T = 1` — sample from the model's actual distribution. This is the honest one:
  it reproduces the training distribution, warts and all.
- `T < 1` — sharpen. More "typical", more confident, less diverse.
- `T > 1` — flatten toward uniform. More diverse, more garbage.

One catch: `BOS` is in the vocab but must never be *emitted*, so force its logit to
`-inf` before sampling. (After training the model assigns it almost no mass anyway
— but at high temperature "almost none" becomes "sometimes", and you would emit a
control token mid-protein.)

### Rep 5 — `sample(model, n, temperature=1.0, L=L)`
Start from a column of `BOS`, generate `L` tokens autoregressively, return a list of
`n` decoded strings (without the leading `BOS`).

In [ ]:
def sample(model, n, temperature=1.0, L=L):
    '''Autoregressively generate n sequences. Returns a list of n strings.'''
    # YOUR CODE HERE
    # hint: idx = torch.full((n, 1), BOS, dtype=torch.long); loop L times:
    #       logits = model(idx)[:, -1, :] / temperature   (last position only)
    #       logits[:, BOS] = float('-inf')                (never emit BOS)
    #       torch.multinomial(F.softmax(logits, -1), 1) -> append with torch.cat
    raise NotImplementedError

# --- checkpoint ---
gen = sample(model, 6)
assert len(gen) == 6 and all(len(s) == L for s in gen), 'wrong count/length'
assert all(c in AA for s in gen for c in s), 'only real amino acids may be emitted'
for s in gen:
    print(s, ' ', ''.join('H' if c in HYDRO else '.' for c in s))
print('\nLook at the H/. columns: a period-7 register the model invented itself ✓')

## Part 7 — protein instance: did it actually learn the biology?

Low perplexity is a claim about *likelihood*. The question a protein designer asks
is different: **do the samples have the property?** Those come apart all the time,
so measure the property directly.

Score a sequence by the best heptad register it fits: for each of the 7 phases,
count positions whose hydrophobicity matches the ideal `a`/`d` pattern, and keep
the best. `1.0` = a perfect amphipathic helix; frequency-matched random sequences
land near `0.67` (with 7 phases to try, chance does better than you'd guess — which
is exactly why you compute the random baseline instead of trusting your intuition).

### Rep 6 — `heptad_score(seq)`
Try all 7 phases; for each, count positions where
`((i - phase) % 7) in CORE` agrees with `seq[i] in HYDRO`; return the best
fraction.

In [ ]:
def heptad_score(seq):
    '''Best-over-phases fraction of positions matching the ideal heptad pattern.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
assert abs(heptad_score('AKKAKKK' * 4) - 1.0) < 1e-9, 'that is a perfect a/d pattern'
train_str = [decode(r) for r in train[:1000]]
rand_str = [''.join(AA[i] for i in rng.choice(N_AA, size=L, p=f / f.sum())) for _ in range(1000)]
gen1000 = sample(model, 1000, temperature=1.0)

hs_train = np.mean([heptad_score(s) for s in train_str])
hs_model = np.mean([heptad_score(s) for s in gen1000])
hs_rand = np.mean([heptad_score(s) for s in rand_str])
assert hs_model > 0.85, 'samples should be strongly amphipathic'
print('heptad score   real data %.3f  |  model %.3f  |  random %.3f'
      % (hs_train, hs_model, hs_rand))
print('\nThe model matches the data, not the random baseline — it learned the rule,')
print('never having been told the period, the phase, or which residues are greasy. ✓')

## Part 8 — temperature is the 0.2 frontier, again

In 0.2 you traded reward against diversity by tilting a fixed pool. Here the same
tradeoff appears from a completely different mechanism — a knob on a learned
generator — and that is the point: it is structural, not an artifact of one method.

Measuring diversity needs care. The obvious metric, *fraction of unique
sequences*, is useless here: the space is `20^28`, so it reads `1.000` at every
temperature and tells you nothing. We use **mean per-position entropy** instead —
the average entropy (nats) of the letter distribution at each position across many
samples. It is in the same units as the loss, and it has a meaningful target: the
**real data's** entropy.

That target is the punchline. `T = 1` should land on *both* of the data's values.
Turning `T` down produces sequences that score **better than real data** on the
heptad metric — unnaturally regular, drawn from an ever-shrinking corner of
sequence space. An oracle would love them; a wet lab would find them boring and
probably insoluble. This is mode collapse, and it is the same failure that
over-optimizing any reward produces.

In [ ]:
def positional_entropy(seqs):
    '''Mean over positions of the entropy (nats) of the letter distribution.'''
    arr = np.array([[STOI[c] for c in s] for s in seqs])
    ent = []
    for t in range(arr.shape[1]):
        p = np.bincount(arr[:, t], minlength=N_AA).astype(float)
        p = p[p > 0] / p.sum()
        ent.append(-(p * np.log(p)).sum())
    return float(np.mean(ent))

Ts = [0.1, 0.3, 0.5, 0.7, 0.85, 1.0, 1.2, 1.5, 2.0, 3.0]
rows = []
for T in Ts:
    g = sample(model, 500, temperature=T)
    rows.append((T, np.mean([heptad_score(s) for s in g]), positional_entropy(g),
                 len(set(g)) / len(g)))

d_hep, d_ent = hs_train, positional_entropy(train_str)
print(' T     heptad   entropy   unique-frac')
for T, hp, en, uq in rows:
    print('%5.2f   %.3f    %.3f     %.3f' % (T, hp, en, uq))
print('real data      %.3f    %.3f' % (d_hep, d_ent))
print('\nunique-frac pinned at 1.000 everywhere — a diversity metric that measures nothing.')

r = np.array(rows)
fig, ax = plt.subplots()
ax.plot(r[:, 1], r[:, 2], '-o', color=BLUE, lw=2, ms=6)
for T, hp, en, _ in rows:                      # label selectively — not every point
    if T in (0.1, 0.3, 0.5, 0.7, 1.0, 2.0, 3.0):
        off, ha = ((-8, -13), 'right') if T == 1.0 else ((5, -10), 'left')
        ax.annotate('T=%g' % T, (hp, en), fontsize=8, color=INK,
                    xytext=off, ha=ha, textcoords='offset points')
ax.plot([d_hep], [d_ent], marker='*', ms=16, color=GREEN, ls='none')
ax.annotate('real data', (d_hep, d_ent), fontsize=9, color=GREEN, fontweight='bold',
            xytext=(0, 13), ha='center', textcoords='offset points')
ax.margins(x=0.13)
ax.set_xlabel('heptad score  (structure / "reward")')
ax.set_ylabel('positional entropy, nats  (diversity)')
ax.set_title('temperature walks the same frontier you drew in 0.2')
ax.grid(alpha=.15)
plt.show()
print('T=1 sits on the data. Below it you buy structure with diversity. ✓')

### A note on reading attention maps

It is tempting to plot the attention weights and narrate a story about period-7
heads. Resist it. Below is the averaged map from layer 0 — the triangular shape
confirms causality (the thing we asserted), but the learned pattern here does
**not** cleanly single out lag 7, even though the model provably uses the register.

That gap is worth sitting with: a network can solve a task without any single head
implementing the textbook algorithm — the computation is spread across heads,
layers, the MLPs, and the residual stream. Attention maps are a hypothesis
generator, never evidence on their own.

In [ ]:
model.eval()
with torch.no_grad():
    model(xv_t[:256])
A = model.blocks[0].attn.last_att.mean(dim=(0, 1)).numpy()   # avg over batch and heads

plt.figure(figsize=(4.4, 3.8))
plt.imshow(A, cmap='Blues', interpolation='nearest')
plt.colorbar(label='mean attention weight')
plt.xlabel('key position (attended to)'); plt.ylabel('query position')
plt.title('attention, layer 0 (averaged)')
plt.show()
print('Everything above the diagonal is exactly zero — that is your mask. ✓')

## Reflection — what just transferred

- **The chain rule is the whole model.** `p(x) = prod_t p(x_t | x_<t)` is exact;
  an autoregressive "model" is just a function approximator for one conditional,
  applied `L` times. Swap amino acids for text tokens and you have GPT.
- **The causal mask *is* the autoregressive property.** One `-inf` before one
  softmax. Get it wrong and you leak the future — with a loss curve that looks
  fantastic while the model is useless.
- **Teacher forcing** turns one sequence into `L` supervised examples, trained in
  parallel — which is why transformers train fast and still generate slowly.
- **Perplexity is only half the story.** You measured likelihood *and* a property
  (`heptad_score`), because a model can win one and lose the other. In Track 3 the
  property oracle is ESMFold instead of a 6-line function; the discipline is identical.
- **Temperature is 0.2's tilting knob wearing a different hat.** Two unrelated
  mechanisms, one frontier. Push reward, lose diversity — always.
- **Know your floor.** Because we synthesized the data we could compute the
  Bayes-optimal perplexity and see the model land within ~2% of it. With real data
  you can't — so build the habit of constructing baselines (uniform, unigram,
  bigram) that bracket the answer.

**Next rep:** `1.2 Masked language model` — drop the left-to-right constraint. Train
by masking random positions (the ESM/BERT objective), then *generate* by Gibbs
sampling from a model that was never designed to be a generator at all.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def make_batch(data, batch_size):
    ix = rng.integers(0, len(data), size=batch_size)
    s = data[ix]
    x = np.concatenate([np.full((batch_size, 1), BOS, dtype=np.int64), s[:, :-1]], axis=1)
    return torch.from_numpy(x), torch.from_numpy(s.copy())

def fit_bigram(data, alpha=1.0):
    x, y = full_xy(data)
    counts = np.full((VOCAB, VOCAB), alpha)
    np.add.at(counts, (x.numpy().ravel(), y.numpy().ravel()), 1.0)
    return counts / counts.sum(axis=1, keepdims=True)

def causal_self_attention(q, k, v):
    d_head = q.size(-1)
    att = (q @ k.transpose(-2, -1)) / math.sqrt(d_head)      # [B, nh, T, T]
    T = q.size(-2)
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=q.device))
    att = att.masked_fill(~mask, float('-inf'))              # mask BEFORE softmax
    att = F.softmax(att, dim=-1)
    return att @ v, att

def perplexity(model, data):
    x, y = full_xy(data)
    model.eval()
    with torch.no_grad():
        nll = F.cross_entropy(model(x).reshape(-1, VOCAB), y.reshape(-1)).item()
    return math.exp(nll)

def sample(model, n, temperature=1.0, L=L):
    model.eval()
    idx = torch.full((n, 1), BOS, dtype=torch.long)
    with torch.no_grad():
        for _ in range(L):
            logits = model(idx)[:, -1, :] / temperature
            logits[:, BOS] = float('-inf')
            nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, nxt], dim=1)
    return [decode(row) for row in idx[:, 1:].numpy()]

def heptad_score(seq):
    best = 0.0
    for phi in range(7):
        m = sum(1 for i, c in enumerate(seq) if (((i - phi) % 7) in CORE) == (c in HYDRO))
        best = max(best, m / len(seq))
    return best

print('reference solutions loaded — re-run the checkpoint cells above')